# GPT-2 Task Decomposition

Fine-tuning GPT-2 to break a high-level task down into actionable subtasks.
Original graduation-project notebook, cleaned up: the dataset now lives in `data/tasks.json`,
and the training / generation code also exists as runnable scripts in `src/`.

## 1. Setup

In [ ]:
import json

import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_cosine_schedule_with_warmup

from src.dataset import TaskDataset

## 2. Dataset

189 high-level tasks with their reference subtasks (1,188 subtasks in total), collected for this project.

In [ ]:
with open("data/tasks.json") as f:
    data = json.load(f)

# Preprocess tasks
tasks = [{"task": item["task"], "subtasks": [subtask["name"] for subtask in item["subtasks"]]} for item in data]
print(len(tasks), "tasks,", sum(len(t["subtasks"]) for t in tasks), "subtasks")
tasks[0]

## 3. Tokenization

Each example becomes one sequence: `task [SEP] subtask [SEP] subtask ...`

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

dataset = TaskDataset(tasks, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

## 4. Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)  # Adjusted learning rate
num_epochs = 3
num_training_steps = num_epochs * len(dataloader)
lr_scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [ ]:
model.train()
for epoch in range(num_epochs):
    for batch in dataloader:
        input_ids, attn_masks = batch
        input_ids = input_ids.to(device)
        attn_masks = attn_masks.to(device)
        outputs = model(input_ids, attention_mask=attn_masks, labels=input_ids)
        loss = outputs.loss
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        print(f"Epoch: {epoch}, Loss: {loss.item()}")

## 5. Save the fine-tuned model

In [ ]:
model.save_pretrained("checkpoints")
tokenizer.save_pretrained("checkpoints")

## 6. Generate subtasks for a new task

In [ ]:
from src.generate import generate_subtasks

task_name = "study english"  # Example task name
generated_subtasks = generate_subtasks(task_name, model, tokenizer, device)
print("Generated Subtasks:", generated_subtasks)